#Import Data

In [1]:

from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [4]:

!cp "/content/drive/MyDrive/Cataract_dataset_classification.v1i.multiclass.zip" .

!unzip Cataract_dataset_classification.v1i.multiclass.zip -d cataract_dataset1/

Streaming output truncated to the last 5000 lines.
 extracting: cataract_dataset1/train/Video17_frame010510_png_jpg.rf.d6b22f3898bab794d06d97b4883a68e6.jpg  
 extracting: cataract_dataset1/train/Video17_frame010520_png_jpg.rf.0a6679683e1fd48a2c89e45b00b08d89.jpg  
 extracting: cataract_dataset1/train/Video17_frame010530_png_jpg.rf.d5926e8eab8bfb3b024b27f9a87a2274.jpg  
 extracting: cataract_dataset1/train/Video17_frame010560_png_jpg.rf.877b6a97df63e137cf06fb31734584bc.jpg  
 extracting: cataract_dataset1/train/Video17_frame010570_png_jpg.rf.b9c4b6c365f1aaf839815e378adf70bf.jpg  
 extracting: cataract_dataset1/train/Video17_frame010590_png_jpg.rf.d38375cecf64947057fea3f2afa5cefa.jpg  
 extracting: cataract_dataset1/train/Video17_frame010620_png_jpg.rf.848ec246aa162d9066714c13524fc26d.jpg  
 extracting: cataract_dataset1/train/Video17_frame010650_png_jpg.rf.c4a99e4060c47ce6cca3d232d95c5ac4.jpg  
 extracting: cataract_dataset1/train/Video17_frame010690_png_jpg.rf.e60b98a706dfd02255ebf2b74

In [5]:
!pip install iterative-stratification

In [6]:
import os
import shutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torchvision import transforms, models
from PIL import Image
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from google.colab import drive

In [7]:
base_path = "/content/cataract_dataset1"
output_folder = "/content/collected_csv"
os.makedirs(output_folder, exist_ok=True)

#Combine train and validation Dataset

In [8]:
base_path = "/content/cataract_dataset1"
output_folder = "/content/collected_csv"
os.makedirs(output_folder, exist_ok=True)

for split in ["train", "valid"]:
    split_path = os.path.join(base_path, split)
    if os.path.exists(split_path):
        for file in os.listdir(split_path):
            if file.endswith(".csv"):
                shutil.copy(os.path.join(split_path, file), os.path.join(output_folder, f"{split}_{file}"))

df_train = pd.read_csv(os.path.join(output_folder, "train__classes.csv"))
df_valid = pd.read_csv(os.path.join(output_folder, "valid__classes.csv"))

df_train.columns = df_train.columns.str.strip()
df_valid.columns = df_valid.columns.str.strip()


df_train['folder'] = os.path.join(base_path, "train")
df_valid['folder'] = os.path.join(base_path, "valid")


df_combined = pd.concat([df_train, df_valid], ignore_index=True)
target_cols = [c for c in df_combined.columns if c not in ['filename', 'folder']]
Y = df_combined[target_cols].values

print(f"📊 Total samples for K-Fold (Train + Valid): {len(df_combined)}")

📊 Total samples for K-Fold (Train + Valid): 6127


#Preprocessing

In [9]:
train_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, hue=0.1, saturation=0.2),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Focal loss class

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        probs = torch.sigmoid(inputs)
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        if self.reduction == 'mean': return focal_loss.mean()
        else: return focal_loss.sum()

In [11]:

class CataractDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.classes = [c for c in self.df.columns if c not in ['filename', 'folder']]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(row['folder'], row['filename'])
        image = Image.open(img_path).convert('RGB')
        labels = torch.tensor(row[self.classes].values.astype(float), dtype=torch.float32)

        if self.transform:
            image = self.transform(image)
        return image, labels

In [12]:
train_dataset = CataractDataset(df_combined, transform=train_transform)
valid_dataset = CataractDataset(df_combined, transform=valid_transform)

def get_model():
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, len(target_cols)) # 12 Classes
    return model.to(device)


#Training model

In [13]:

K_FOLDS = 5
EPOCHS_PER_FOLD = 5
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

mskf = MultilabelStratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = {}

print(f"\n Starting {K_FOLDS}-Fold Cross Validation with Online Augmentation...")

for fold, (train_idx, val_idx) in enumerate(mskf.split(np.zeros(len(df_combined)), Y)):
    print(f"\n{'-'*30}\n🔄 FOLD {fold + 1}/{K_FOLDS}\n{'-'*30}")


    train_subsampler = SubsetRandomSampler(train_idx)
    val_subsampler = SubsetRandomSampler(val_idx)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_subsampler, num_workers=2)
    val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, sampler=val_subsampler, num_workers=2)


    model = get_model()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = FocalLoss(gamma=2)

    best_fold_acc = 0.0

    for epoch in range(EPOCHS_PER_FOLD):

        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_correct += (preds == labels).sum().item()
            train_total += labels.numel()

        avg_train_loss = train_loss / len(train_idx)
        train_acc = train_correct / train_total


        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.numel()

        avg_val_loss = val_loss / len(val_idx)
        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f}, Acc: {val_acc:.4f}")

        if val_acc > best_fold_acc:
            best_fold_acc = val_acc

    fold_results[f'Fold {fold+1}'] = best_fold_acc


    del model, optimizer, criterion
    torch.cuda.empty_cache()



 Starting 5-Fold Cross Validation with Online Augmentation...

------------------------------
🔄 FOLD 1/5
------------------------------
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 103MB/s]


Epoch 1 | Train Loss: 0.0613, Acc: 0.9169 | Val Loss: 0.0320, Acc: 0.9617
Epoch 2 | Train Loss: 0.0329, Acc: 0.9575 | Val Loss: 0.0224, Acc: 0.9754
Epoch 3 | Train Loss: 0.0252, Acc: 0.9674 | Val Loss: 0.0174, Acc: 0.9800
Epoch 4 | Train Loss: 0.0211, Acc: 0.9724 | Val Loss: 0.0155, Acc: 0.9817
Epoch 5 | Train Loss: 0.0183, Acc: 0.9763 | Val Loss: 0.0148, Acc: 0.9830

------------------------------
🔄 FOLD 2/5
------------------------------
Epoch 1 | Train Loss: 0.0653, Acc: 0.9098 | Val Loss: 0.0371, Acc: 0.9532
Epoch 2 | Train Loss: 0.0349, Acc: 0.9535 | Val Loss: 0.0224, Acc: 0.9734
Epoch 3 | Train Loss: 0.0259, Acc: 0.9664 | Val Loss: 0.0182, Acc: 0.9788
Epoch 4 | Train Loss: 0.0218, Acc: 0.9719 | Val Loss: 0.0157, Acc: 0.9819
Epoch 5 | Train Loss: 0.0184, Acc: 0.9761 | Val Loss: 0.0155, Acc: 0.9819

------------------------------
🔄 FOLD 3/5
------------------------------
Epoch 1 | Train Loss: 0.0617, Acc: 0.9158 | Val Loss: 0.0358, Acc: 0.9559
Epoch 2 | Train Loss: 0.0330, Acc: 0.9

#K-Fold Validation Results

In [15]:

print("\n📊 --- K-Fold Validation Results ---")
accuracies = list(fold_results.values())

for key, value in fold_results.items():
    print(f"{key}: {value:.4f}")

mean_acc = np.mean(accuracies)
variance_acc = np.var(accuracies)
std_acc = np.std(accuracies)

print("-" * 30)
print(f"📈 Mean Accuracy: {mean_acc:.4f}")
print(f"📉 Variance: {variance_acc:.6f}")
print(f"📏 Standard Deviation: {std_acc:.6f}")

if std_acc < 0.05:
    print("\n✅ The standard deviation is low. Our architecture is highly robust against changes in the training data.")
else:
    print("\n⚠️ The standard deviation is high. The model's stability relative to the data distribution is low.")


📊 --- K-Fold Validation Results ---
Fold 1: 0.9830
Fold 2: 0.9819
Fold 3: 0.9815
Fold 4: 0.9833
Fold 5: 0.9807
------------------------------
📈 Mean Accuracy: 0.9821
📉 Variance: 0.000001
📏 Standard Deviation: 0.000953

✅ The standard deviation is low. Our architecture is highly robust against changes in the training data.
